# A6 · PySAL — spatial statistics: is the pattern real?

**What PySAL is:** the *Python Spatial Analysis Library*, a family of packages for **spatial statistics**:

| Package | Job | Daily question |
|---|---|---|
| `libpysal` | **spatial weights**: who is whose neighbour | "Which houses are next door to mine?" |
| `mapclassify` | split values into classes for maps | "Where do I draw the line between 'low' and 'high'?" |
| `esda` | exploratory spatial data analysis: Moran's I, hot spots | "Are high values clustered, or spread by chance?" |
| `spreg` | spatial regression | "What explains accidents, if neighbours influence each other?" |
| `pointpats` | point-pattern analysis | "Are these points clustered, regular or random?" |

**Why it matters:** in A3 you asked "is it real or chance?" with a home-made Monte Carlo test. PySAL gives you the
standard, published tools for that question. They are the tools you will cite in a PhD or a paper.

**The one big idea — Tobler's first law of geography:** *"Everything is related to everything else, but near things
are more related than distant things."* Daily picture: house prices in your street are more like your neighbour's
than like prices across town. Spatial statistics measure **how strong** that "near is similar" effect is.

**Where this fits in your plan:** right after GeoPandas. It turns the **statistical** question type into proper methods.
Riverton's 9 neighbourhoods are too few for statistics, so we work on a **grid of 500 m cells** (144 cells).

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))      # lets the notebook find the `geotrain` helper
warnings.filterwarnings("ignore")
from geotrain import write_riverton, DATA_DIR
if not (DATA_DIR / "riverton.gpkg").exists():   # build the Riverton data the first time
    write_riverton()
GPKG = DATA_DIR / "riverton.gpkg"
print("Data folder:", DATA_DIR)

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box

def make_grid(size):
    """Square grid over Riverton with accidents, residents and primary-road length per cell."""
    cells = [box(x, y, x + size, y + size)
             for y in range(5_818_000, 5_824_000, size) for x in range(390_000, 396_000, size)]
    g = gpd.GeoDataFrame({"cell": range(len(cells))}, geometry=cells, crs=32633)
    acc = gpd.sjoin(L["accidents"], g, predicate="within").groupby("cell").size()
    res = gpd.sjoin(L["houses"], g, predicate="within").groupby("cell")["residents"].sum()
    prim = gpd.clip(L["roads"].query("road_type == 'primary'"), g.union_all())
    road = gpd.overlay(prim[["geometry"]], g, how="intersection", keep_geom_type=True)
    g["accidents"] = g["cell"].map(acc).fillna(0)
    g["residents"] = g["cell"].map(res).fillna(0).astype(float)
    g["road_km"] = g["cell"].map(road.assign(km=road.length / 1000).groupby("cell")["km"].sum()).fillna(0)
    return g

L = {n: gpd.read_file(GPKG, layer=n) for n in ["accidents", "houses", "roads", "sensors", "neighbourhoods"]}
grid = make_grid(500)                     # 12 x 12 = 144 cells of 500 m
print(len(grid), "cells;", int(grid.accidents.sum()), "accidents")

In [ ]:
import libpysal
from libpysal import weights
import esda
import mapclassify
import spreg
import pointpats
ax = grid.plot(column="accidents", cmap="Reds", legend=True, edgecolor="white", figsize=(5, 5))
L["roads"].plot(ax=ax, color="black", linewidth=.6); ax.set_title("accidents per 500 m cell"); plt.show()

---
## Level 1 — Basics: spatial weights — who is my neighbour?

**Goal of this level:** build the neighbour lists (weights) that every spatial statistic needs.

**Daily-life picture:** Before asking 'are neighbours similar?', you must agree who counts as a neighbour: the house next door? the whole street? everyone within 1 km?

### Exercise 1.1 — Queen contiguity

**Step 1 · Purpose.** What is the **purpose** of `weights.Queen.from_dataframe(gdf)`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`w = weights.Queen.from_dataframe(grid, use_index=False)`. `w.n` = number of units, `w.mean_neighbors`, `w.neighbors[0]` = neighbours of cell 0 (a corner cell).

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
wq = weights.____.from_dataframe(grid, use_index=False)
print(wq.n, "cells | mean neighbours:", round(wq.mean_neighbors, 2))
print("neighbours of corner cell 0:", wq.neighbors[____])
print("neighbours of cell 13 (inside):", wq.neighbors[13])

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Makes a neighbour list where two polygons are neighbours if they share an edge **or a corner** (like the queen in chess).

**Real life:** Defining neighbouring districts for a 'do rich districts border rich districts?' analysis.

```python
wq = weights.Queen.from_dataframe(grid, use_index=False)
print(wq.n, "cells | mean neighbours:", round(wq.mean_neighbors, 2))
print("neighbours of corner cell 0:", wq.neighbors[0])
print("neighbours of cell 13 (inside):", wq.neighbors[13])
```

Inside cells have 8 queen neighbours, edge cells 5, corner cells 3.

</details>

### Exercise 1.2 — Rook contiguity

**Step 1 · Purpose.** What is the **purpose** of `weights.Rook.from_dataframe(gdf)`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Same call with `Rook`. Compare the neighbour count of cell 13: queen 8, rook 4.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
wr = weights.____.from_dataframe(grid, use_index=False)
print(len(wq.neighbors[13]), "queen vs", len(wr.neighbors[13]), "rook neighbours")

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Neighbours only if they share an **edge** (like the rook in chess): no corner neighbours.

**Real life:** Used when touching at a single point should not count (e.g. two fields that only meet at a corner).

```python
wr = weights.Rook.from_dataframe(grid, use_index=False)
print(len(wq.neighbors[13]), "queen vs", len(wr.neighbors[13]), "rook neighbours")
```

</details>

### Exercise 1.3 — K nearest neighbours for points

**Step 1 · Purpose.** What is the **purpose** of `weights.KNN.from_dataframe(points, k=4)`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`weights.KNN.from_dataframe(L["sensors"], k=4)`. Look at `w.neighbors[0]`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
wk = weights.KNN.from_dataframe(L["sensors"], k=____)
print(wk.n, "sensors, neighbours of sensor 0:", wk.neighbors[0])

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Each point's neighbours are its k closest points. Everyone gets the same number of neighbours.

**Real life:** Air-quality sensors are points, not touching polygons; 'my 4 nearest sensors' is a natural neighbourhood.

```python
wk = weights.KNN.from_dataframe(L["sensors"], k=4)
print(wk.n, "sensors, neighbours of sensor 0:", wk.neighbors[0])
```

</details>

### Exercise 1.4 — Distance band

**Step 1 · Purpose.** What is the **purpose** of `weights.DistanceBand.from_dataframe(points, threshold=d)`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`threshold=1500`, `binary=True`. Check `w.islands` (points with no neighbour) and `w.cardinalities` (number of neighbours per point).

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
wd = weights.DistanceBand.from_dataframe(L["sensors"], threshold=____, binary=True, silence_warnings=True)
print("islands:", wd.islands)
print("neighbours per sensor:", list(wd.cardinalities.values()))

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Neighbours are all points within distance d. Some points may have many neighbours, some none ('islands').

**Real life:** 'Shops within 1 km compete with each other' — a fixed distance matters more than a fixed count.

```python
wd = weights.DistanceBand.from_dataframe(L["sensors"], threshold=1500, binary=True, silence_warnings=True)
print("islands:", wd.islands)
print("neighbours per sensor:", list(wd.cardinalities.values()))
```

**Recommendation:** polygons that tile the space → Queen; points → KNN (no islands). Always say which weights you used; results depend on it.

</details>

---
## Level 2 — Core tools: spatial lag and map classes

**Goal of this level:** compute 'the average of my neighbours' and classify values honestly for maps.

**Daily-life picture:** Your street's reputation is the average of the houses around you. And the colour breaks on a map can make the same data look calm or alarming.

### Exercise 2.1 — Row-standardise the weights

**Step 1 · Purpose.** What is the **purpose** of `w.transform = 'r'`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Set `wq.transform = "r"`. Check: the weights of cell 0 are 1/3 each.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
wq.transform = "____"
print(wq.weights[0])

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Rescales each row of weights so they sum to 1; then 'neighbour sum' becomes 'neighbour average'.

**Real life:** Needed so a cell with 8 neighbours is not treated as '8 times more influenced' than a corner cell with 3.

```python
wq.transform = "r"
print(wq.weights[0])
```

</details>

### Exercise 2.2 — Spatial lag: the average of my neighbours

**Step 1 · Purpose.** What is the **purpose** of `weights.lag_spatial(w, y)`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`grid["acc_lag"] = weights.lag_spatial(wq, grid["accidents"])`. Compare the two maps side by side.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
grid["acc_lag"] = weights.____(wq, grid["accidents"])
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
grid.plot(column="accidents", cmap="Reds", ax=axs[0], legend=True); axs[0].set_title("own value")
grid.plot(column="acc_lag", cmap="Reds", ax=axs[1], legend=True); axs[1].set_title("average of neighbours")
plt.show()

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** For every unit, computes the (weighted) average of its neighbours' values.

**Real life:** 'How dangerous is the area around this cell?' — smoother than a single cell's count.

```python
grid["acc_lag"] = weights.lag_spatial(wq, grid["accidents"])
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
grid.plot(column="accidents", cmap="Reds", ax=axs[0], legend=True); axs[0].set_title("own value")
grid.plot(column="acc_lag", cmap="Reds", ax=axs[1], legend=True); axs[1].set_title("average of neighbours")
plt.show()
```

</details>

### Exercise 2.3 — Map classes: quantiles vs natural breaks

**Step 1 · Purpose.** What is the **purpose** of `mapclassify.Quantiles(y, k=5) / NaturalBreaks`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Print both classifiers; they show the class borders and counts. With many zeros, quantiles struggle (several classes share the value 0).

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
y = grid["accidents"]
print(mapclassify.____(y, k=5))
print(mapclassify.____(y, k=5))

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Splits values into classes: quantiles = same **number** of units per class; natural breaks = class borders at big jumps in the data.

**Real life:** Choosing breaks for a public map: the same data can look 'everywhere bad' or 'a few hot spots' depending on the method.

```python
y = grid["accidents"]
print(mapclassify.Quantiles(y, k=5))
print(mapclassify.NaturalBreaks(y, k=5))
```

</details>

### Exercise 2.4 — Classified choropleth in one line

**Step 1 · Purpose.** What is the **purpose** of `gdf.plot(column=..., scheme='NaturalBreaks', k=5)`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Try `scheme="NaturalBreaks"` and `scheme="EqualInterval"`, `legend=True`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
grid.plot(column="accidents", scheme="____", k=5, cmap="Reds", legend=True, ax=axs[0]); axs[0].set_title("natural breaks")
grid.plot(column="accidents", scheme="EqualInterval", k=5, cmap="Reds", legend=True, ax=axs[1]); axs[1].set_title("equal interval")
plt.show()

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** GeoPandas calls mapclassify for you when you pass `scheme=`.

**Real life:** The standard way to make publication maps with honest class breaks.

```python
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
grid.plot(column="accidents", scheme="NaturalBreaks", k=5, cmap="Reds", legend=True, ax=axs[0]); axs[0].set_title("natural breaks")
grid.plot(column="accidents", scheme="EqualInterval", k=5, cmap="Reds", legend=True, ax=axs[1]); axs[1].set_title("equal interval")
plt.show()
```

</details>

---
## Level 3 — Combining: global and local spatial autocorrelation

**Goal of this level:** measure clustering for the whole town (Moran's I), then find where the clusters are (LISA, Getis-Ord).

**Daily-life picture:** First the doctor asks 'do you have a fever?' (global). Then 'where does it hurt?' (local).

### Exercise 3.1 — Global Moran's I

**Step 1 · Purpose.** What is the **purpose** of `esda.Moran(y, w)`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`mi = esda.Moran(grid["accidents"], wq)`, then `mi.I`, `mi.EI` (expected under randomness ≈ −1/(n−1)), `mi.p_sim`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
mi = esda.____(grid["accidents"], wq)
print(f"I = {mi.I:.3f}, expected if random = {mi.EI:.3f}, p = {mi.p_sim}")

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** One number for the whole map: I > 0 means similar values cluster, I ≈ 0 means random, I < 0 means a checkerboard. `p_sim` says how likely such an I is by chance (permutation test).

**Real life:** 'Do high-accident cells sit next to other high-accident cells?' before deciding if a local hot-spot search makes sense.

```python
mi = esda.Moran(grid["accidents"], wq)
print(f"I = {mi.I:.3f}, expected if random = {mi.EI:.3f}, p = {mi.p_sim}")
```

The permutation test is the Monte Carlo idea from A3: shuffle the values over the cells 999 times and see how often I is as big.

</details>

### Exercise 3.2 — Sanity check: shuffle the map

**Step 1 · Purpose.** What is the **purpose** of computing Moran's I on shuffled values?

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`np.random.default_rng(0).permutation(grid["accidents"].values)`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
shuffled = np.random.default_rng(0).____(grid["accidents"].values)
m0 = esda.Moran(shuffled, wq)
print(f"shuffled: I = {m0.I:.3f}, p = {m0.p_sim}")

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Shuffling destroys any spatial pattern, so I should drop near 0 and p should be large. It proves you understand what Moran's I measures.

**Real life:** A good habit: test your method on data where you know the answer.

```python
shuffled = np.random.default_rng(0).permutation(grid["accidents"].values)
m0 = esda.Moran(shuffled, wq)
print(f"shuffled: I = {m0.I:.3f}, p = {m0.p_sim}")
```

</details>

### Exercise 3.3 — Local Moran (LISA): where are the clusters?

**Step 1 · Purpose.** What is the **purpose** of `esda.Moran_Local(y, w)`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`lisa = esda.Moran_Local(grid["accidents"], wq, seed=1)`. Significant hot spots: `(lisa.q == 1) & (lisa.p_sim < 0.05)`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
lisa = esda.____(grid["accidents"], wq, seed=1)
labels = {1: "HH hot spot", 2: "LH", 3: "LL cold spot", 4: "HL outlier"}
grid["lisa"] = [labels[q] if p < 0.05 else "not significant" for q, p in zip(lisa.q, lisa.p_sim)]
print(grid["lisa"].value_counts())
ax = grid.plot(column="lisa", categorical=True, legend=True, cmap="Set1", figsize=(6, 6), edgecolor="white")
L["roads"].plot(ax=ax, color="black", linewidth=.6); plt.show()

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** One statistic per unit. With `q` (quadrant) it labels each unit: 1 = High-High (hot spot), 2 = Low-High, 3 = Low-Low (cold spot), 4 = High-Low (outlier); `p_sim` says which are significant.

**Real life:** Finding the exact streets where accidents cluster, to place speed cameras.

```python
lisa = esda.Moran_Local(grid["accidents"], wq, seed=1)
labels = {1: "HH hot spot", 2: "LH", 3: "LL cold spot", 4: "HL outlier"}
grid["lisa"] = [labels[q] if p < 0.05 else "not significant" for q, p in zip(lisa.q, lisa.p_sim)]
print(grid["lisa"].value_counts())
ax = grid.plot(column="lisa", categorical=True, legend=True, cmap="Set1", figsize=(6, 6), edgecolor="white")
L["roads"].plot(ax=ax, color="black", linewidth=.6); plt.show()
```

</details>

### Exercise 3.4 — Getis-Ord Gi*: hot spots by intensity

**Step 1 · Purpose.** What is the **purpose** of `esda.G_Local(y, w, star=True)`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Gi* needs **binary** weights (1 = neighbour, 0 = not): pass `transform="B"` — without it, esda row-standardises silently. `g.Zs` are z-scores; |z| > 1.96 ≈ significant at 5 %.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
wb = weights.Queen.from_dataframe(grid, use_index=False)
g = esda.G_Local(grid["accidents"], wb, transform="____", star=True, seed=1)
grid["gi_z"] = g.Zs
print((grid["gi_z"] > 1.96).sum(), "hot cells,", (grid["gi_z"] < -1.96).sum(), "cold cells")
grid.plot(column="gi_z", cmap="coolwarm", legend=True, vmin=-4, vmax=4, figsize=(5, 5)); plt.show()

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** For each unit, compares the sum of values around it (including itself) with the town average. Large positive z = hot spot, large negative = cold spot.

**Real life:** The classic 'hot-spot analysis' in crime and health studies (ArcGIS/QGIS call it Getis-Ord Gi*).

```python
wb = weights.Queen.from_dataframe(grid, use_index=False)
g = esda.G_Local(grid["accidents"], wb, transform="B", star=True, seed=1)
grid["gi_z"] = g.Zs
print((grid["gi_z"] > 1.96).sum(), "hot cells,", (grid["gi_z"] < -1.96).sum(), "cold cells")
grid.plot(column="gi_z", cmap="coolwarm", legend=True, vmin=-4, vmax=4, figsize=(5, 5)); plt.show()
```

</details>

---
## Level 4 — Professional: statistical and modelling questions

**Goal of this level:** handle multiple testing, the areal unit problem, spatial regression, and point patterns.

**Daily-life picture:** A careful scientist asks: would I get the same answer with a different grid? Am I fooled by testing 144 cells at once? Does my model leave a pattern behind?

### Exercise 4.1 — How many hot spots survive a stricter test?

**Scenario.** With 144 cells tested at p < 0.05, about 7 cells could look significant **by pure chance** (144 × 0.05). Use the False Discovery Rate (FDR) on the Gi* p-values (`g.p_norm`) to set a stricter threshold. How many hot cells remain?

**Step 1 · Question type.** What *kind* of question is this? (descriptive, measurement, proximity, overlay, statistical, modelling, decision, temporal…)

**Step 2 · Plan.** Write the commands you need **in order**, in plain words first.

**Step 3 · Code.** Turn the plan into code below.

<details><summary>💡 <b>Hint</b></summary>

`thr = esda.fdr(g.p_norm, 0.05)` returns a stricter p-threshold. Hot and significant = `(g.Zs > 0) & (g.p_norm <= thr)`. Daily picture: if 144 people each flip a coin 5 times, someone will get 5 heads — that is not magic.

</details>

✍️ **Your answers:**

- Question type: …
- Plan (steps): 1. … 2. … 3. …
- Result / interpretation: …

In [ ]:
thr = esda.____(g.p_norm, 0.05)
loose = (g.Zs > 0) & (g.p_norm < 0.05)
strict = (g.Zs > 0) & (g.p_norm <= thr)
print(f"p < 0.05: {loose.sum()} hot cells | FDR (p <= {thr:.4f}): {strict.sum()} hot cells")
ax = grid.plot(color="lightgrey", edgecolor="white", figsize=(5, 5)); grid[strict].plot(ax=ax, color="red")
L["roads"].plot(ax=ax, color="black", linewidth=.6); ax.set_title("hot spots after FDR"); plt.show()

<details><summary>✅ <b>Solution</b></summary>

🧭 **Question type:** Statistical (multiple testing)

Local statistics test many places at once, so some 'hot spots' are chance. The cells that survive FDR are the ones worth acting on — in Riverton they sit on the dangerous junctions.
Note: esda also gives permutation p-values (`g.p_sim`). For skewed counts they are more cautious; report which p-values you used.

```python
thr = esda.fdr(g.p_norm, 0.05)
loose = (g.Zs > 0) & (g.p_norm < 0.05)
strict = (g.Zs > 0) & (g.p_norm <= thr)
print(f"p < 0.05: {loose.sum()} hot cells | FDR (p <= {thr:.4f}): {strict.sum()} hot cells")
ax = grid.plot(color="lightgrey", edgecolor="white", figsize=(5, 5)); grid[strict].plot(ax=ax, color="red")
L["roads"].plot(ax=ax, color="black", linewidth=.6); ax.set_title("hot spots after FDR"); plt.show()
```

</details>

### Exercise 4.2 — Does the answer change with the grid size? (MAUP)

**Scenario.** Compute global Moran's I for accidents on grids of **250 m, 500 m and 1000 m**. Does the strength of clustering change?

**Step 1 · Question type.** What *kind* of question is this? (descriptive, measurement, proximity, overlay, statistical, modelling, decision, temporal…)

**Step 2 · Plan.** Write the commands you need **in order**, in plain words first.

**Step 3 · Code.** Turn the plan into code below.

<details><summary>💡 <b>Hint</b></summary>

Loop over sizes: `make_grid(size)`, Queen weights row-standardised, `esda.Moran`. Keep I and p in a small table.

</details>

✍️ **Your answers:**

- Question type: …
- Plan (steps): 1. … 2. … 3. …
- Result / interpretation: …

In [ ]:
rows = []
for size in [250, 500, 1000]:
    gg = make_grid(size)
    ww = weights.Queen.from_dataframe(gg, use_index=False); ww.transform = "r"
    m = esda.Moran(gg["accidents"], ww)
    rows.append({"cell_m": size, "cells": len(gg), "I": round(m.I, 3), "p": m.p_sim})
pd.DataFrame(rows)

<details><summary>✅ <b>Solution</b></summary>

🧭 **Question type:** Statistical (modifiable areal unit problem)

I changes with the cell size: the result is partly an effect of how you cut space. This is the **Modifiable Areal Unit Problem** (MAUP).
**Recommendation:** choose the unit from the process (accidents happen on streets → street segments, or a size that matches the planning decision), and report a sensitivity check like this one.

```python
rows = []
for size in [250, 500, 1000]:
    gg = make_grid(size)
    ww = weights.Queen.from_dataframe(gg, use_index=False); ww.transform = "r"
    m = esda.Moran(gg["accidents"], ww)
    rows.append({"cell_m": size, "cells": len(gg), "I": round(m.I, 3), "p": m.p_sim})
pd.DataFrame(rows)
```

</details>

### Exercise 4.3 — What explains accidents? OLS vs spatial regression

**Scenario.** Model accidents per cell from `road_km` (primary road length) and `residents`. First fit an ordinary regression (OLS).
Check whether its **residuals** are still spatially clustered (Moran's I of residuals). If yes, fit a **spatial lag model** and compare.

**Step 1 · Question type.** What *kind* of question is this? (descriptive, measurement, proximity, overlay, statistical, modelling, decision, temporal…)

**Step 2 · Plan.** Write the commands you need **in order**, in plain words first.

**Step 3 · Code.** Turn the plan into code below.

<details><summary>💡 <b>Hint</b></summary>

`y = grid[["accidents"]].values`, `X = grid[["road_km", "residents"]].values`.
`ols = spreg.OLS(y, X, w=wq, spat_diag=True, moran=True, name_y=..., name_x=[...])` → `ols.moran_res` = (I, z, p).
`lag = spreg.ML_Lag(y, X, w=wq, name_y=..., name_x=[...])` → `lag.rho` = how much a cell depends on its neighbours.
Compare fit with `ols.aic` vs `lag.aic` (lower = better).

</details>

✍️ **Your answers:**

- Question type: …
- Plan (steps): 1. … 2. … 3. …
- Result / interpretation: …

In [ ]:
y = grid[["accidents"]].values
X = grid[["road_km", "residents"]].values
ols = spreg.OLS(y, X, w=wq, spat_diag=True, moran=True, name_y="accidents", name_x=["road_km", "residents"])
print("OLS coefficients:", ols.betas.ravel().round(3), "| residual Moran I, p:", np.round(ols.moran_res[0], 3), round(ols.moran_res[2], 4))
lag = spreg.____(y, X, w=wq, name_y="accidents", name_x=["road_km", "residents"])
print("lag rho:", round(float(lag.rho), 3), "| AIC OLS:", round(ols.aic, 1), " AIC lag:", round(lag.aic, 1))

<details><summary>✅ <b>Solution</b></summary>

🧭 **Question type:** Modelling (spatial regression)

Road length is the main driver (each km of primary road adds several accidents per cell). If OLS residuals are still clustered, OLS breaks its assumption of independent errors,
so its p-values are too optimistic. The spatial lag model adds 'the neighbours' accidents' as an explanation (`rho`). Compare AIC and the coefficients.
Print `ols.summary` or `lag.summary` for the full report (with the LM tests that help choose between lag and error models).

```python
y = grid[["accidents"]].values
X = grid[["road_km", "residents"]].values
ols = spreg.OLS(y, X, w=wq, spat_diag=True, moran=True, name_y="accidents", name_x=["road_km", "residents"])
print("OLS coefficients:", ols.betas.ravel().round(3), "| residual Moran I, p:", np.round(ols.moran_res[0], 3), round(ols.moran_res[2], 4))
lag = spreg.ML_Lag(y, X, w=wq, name_y="accidents", name_x=["road_km", "residents"])
print("lag rho:", round(float(lag.rho), 3), "| AIC OLS:", round(ols.aic, 1), " AIC lag:", round(lag.aic, 1))
```

**Why this matters:** Spatial regression is the standard answer to 'which factors explain this spatial pattern?' when neighbours influence each other.

</details>

### Exercise 4.4 — Are accidents clustered as points? (Ripley's K)

**Scenario.** Without any grid: are the 260 accident points more clustered than random points in the same area, and at which distances?

**Step 1 · Question type.** What *kind* of question is this? (descriptive, measurement, proximity, overlay, statistical, modelling, decision, temporal…)

**Step 2 · Plan.** Write the commands you need **in order**, in plain words first.

**Step 3 · Code.** Turn the plan into code below.

<details><summary>💡 <b>Hint</b></summary>

`pointpats.k_test(coords, n_simulations=99, keep_simulations=True)` compares Ripley's K with random patterns. Result: `support` (distances), `statistic` (K), `pvalue` per distance. Small p at a distance = clustered at that scale.

</details>

✍️ **Your answers:**

- Question type: …
- Plan (steps): 1. … 2. … 3. …
- Result / interpretation: …

In [ ]:
coords = np.column_stack([L["accidents"].geometry.x, L["accidents"].geometry.y])
res = pointpats.____(coords, support=np.linspace(50, 1500, 12), n_simulations=99, keep_simulations=True)
sims = res.simulations
plt.plot(res.support, res.statistic, "r-", label="accidents")
plt.fill_between(res.support, sims.min(axis=0), sims.max(axis=0), color="grey", alpha=.4, label="random envelope")
plt.xlabel("distance (m)"); plt.ylabel("K(d)"); plt.legend(); plt.show()
print(pd.DataFrame({"d": res.support.round(), "p": res.pvalue.round(3)}).T)

<details><summary>✅ <b>Solution</b></summary>

🧭 **Question type:** Statistical (point-pattern analysis)

If the red line is above the grey envelope, accidents are more clustered than random at that distance. No grid = no MAUP. Same caution as before: 'random' here means anywhere in the box, not only on streets.

```python
coords = np.column_stack([L["accidents"].geometry.x, L["accidents"].geometry.y])
res = pointpats.k_test(coords, support=np.linspace(50, 1500, 12), n_simulations=99, keep_simulations=True)
sims = res.simulations
plt.plot(res.support, res.statistic, "r-", label="accidents")
plt.fill_between(res.support, sims.min(axis=0), sims.max(axis=0), color="grey", alpha=.4, label="random envelope")
plt.xlabel("distance (m)"); plt.ylabel("K(d)"); plt.legend(); plt.show()
print(pd.DataFrame({"d": res.support.round(), "p": res.pvalue.round(3)}).T)
```

</details>

---
## 🏁 Final test

For each: **question type → which weights / method and why → code → interpretation (including one limitation).**

### Test 1 — 📊 Statistical question

**A.** Is air pollution (PM2.5) at the 18 sensors spatially autocorrelated? Choose suitable weights.

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

Points → KNN weights (no islands). Only 18 sensors, so use few neighbours.
```python
wk4 = weights.KNN.from_dataframe(L["sensors"], k=4); wk4.transform = "r"
m = esda.Moran(L["sensors"]["pm25"], wk4)
print(round(m.I, 3), m.p_sim)
```
With 18 points, the test has little power: a non-significant p does **not** prove there is no pattern.

</details>

### Test 2 — 🛠️ Task question

**B.** Find the **residents** hot spots (where many people live) on the 500 m grid with Getis-Ord Gi*, and list the 5 cells with the highest z-score.

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

```python
gr = esda.G_Local(grid["residents"], wb, transform="B", star=True, seed=1)
print(grid.assign(z=gr.Zs).nlargest(5, "z")[["cell", "residents", "z"]])
```

</details>

### Test 3 — 🗺️ Decision question

**C.** The police can install **3 speed cameras**. Using your analysis, where would you put them and why? What extra data would you want first?

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

Pick cells that are **significant HH hot spots after FDR** (4.1) **and** lie on primary roads; among them, choose those with the most serious/fatal accidents.
Extra data: traffic volume (accidents per vehicle-km is fairer than raw counts), speed measurements, accident causes.
Recommendation: rank by rate (per traffic) rather than by count, because busy roads have more accidents simply because more cars pass.

</details>

### Test 4 — 🧠 Modelling question

**D · Modelling.** For your own research: you want to check whether **poor health clusters** in a city and **what explains it** (income, green space, pollution).
Describe the full modelling plan with PySAL, step by step.

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

1. Units: small areas (census tracts) or a grid; justify the choice (MAUP) and plan a sensitivity check.
2. Weights: Queen for areas (or KNN); row-standardise.
3. Map with honest classes (mapclassify), then global Moran's I on the health rate. For rates with small populations use `esda.Moran_Rate` (it corrects unstable rates).
4. LISA / Gi* for local clusters; FDR correction.
5. OLS with income, green space (NDVI from A5), pollution; test the residuals with Moran's I and the LM tests.
6. If there is spatial dependence: spatial lag (neighbours' health influences) or spatial error (unmeasured spatial factors) model; compare AIC and interpret.
7. Report limits: ecological fallacy (area results do not describe individuals), MAUP, causality.

</details>

### 🪞 Reflection — your long-term plan

Which spatial-statistics question appears in your PhD topic? Write it in one sentence, and name the weights and the method you would start with.

✍️ **Your answer:** …